# Numerisk derivasjon

```{admonition} Læringsutbytte
Etter å ha arbeidet med dette temaet, skal du kunne:

1. forklare forskjellen på analytisk og numerisk derivasjon
2. bruke både notasjonen $f'(x)$ og $\frac{df}{dx}$
3. implementere framover-, bakover- og sentraldifferansen
4. undersøke hvordan steglengde og avrundingsfeil påvirker resultatet
5. derivere diskrete, eksperimentelle data
6. bruke numerisk derivasjon til å tolke titrerkurver og konsentrasjon–tid-data
7. bruke `np.gradient` når det er hensiktsmessig
```

## Derivasjon som endring

Fra videregående kjenner du den deriverte skrevet som $f'(x)$. I naturvitenskapene er Leibniz-notasjonen også svært vanlig:

$$f'(x)=\frac{df}{dx}.$$

Begge uttrykkene beskriver den momentane endringen i $f$ når $x$ endres.

Definisjonen er

$$f'(x)=\frac{df}{dx}=\lim_{\Delta x\rightarrow 0}\frac{f(x+\Delta x)-f(x)}{\Delta x}.$$

I kjemi kan den deriverte blant annet beskrive:

- reaksjonsfart, $\frac{dc}{dt}$
- hvor raskt pH endrer seg i en titrering, $\frac{d\mathrm{pH}}{dV}$
- endringen i et spektroskopisk signal med bølgelengde
- stigningen på en energikurve

På en datamaskin kan vi ikke bruke en uendelig liten $\Delta x$. Vi bruker derfor en liten, men endelig steglengde.


## Framoverdifferansen

Den enkleste tilnærmingen er

$$f'(x)\approx\frac{f(x+\Delta x)-f(x)}{\Delta x}.$$

Dette kalles **framoverdifferansen**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def forward_difference(f, x, dx=1e-6):
    return (f(x + dx) - f(x)) / dx

def f(x):
    return x**2

print(forward_difference(f, 1.0))


For $f(x)=x^2$ er den analytiske deriverte $f'(x)=2x$, så $f'(1)=2$. Den numeriske verdien bør ligge nær dette.

## Bakover- og sentraldifferansen

Bakoverdifferansen bruker punktet bak $x$:

$$f'(x)\approx\frac{f(x)-f(x-\Delta x)}{\Delta x}.$$

Sentraldifferansen bruker ett punkt på hver side:

$$f'(x)\approx\frac{f(x+\Delta x)-f(x-\Delta x)}{2\Delta x}.$$

Sentraldifferansen er vanligvis mer nøyaktig for samme steglengde.


In [ ]:
def backward_difference(f, x, dx=1e-6):
    return (f(x) - f(x - dx)) / dx

def central_difference(f, x, dx=1e-6):
    return (f(x + dx) - f(x - dx)) / (2*dx)

print("Forward:", forward_difference(np.sin, 1.0))
print("Backward:", backward_difference(np.sin, 1.0))
print("Central:", central_difference(np.sin, 1.0))
print("Exact:", np.cos(1.0))


## Hvor liten bør $\Delta x$ være?

Det er fristende å tenke at mindre $\Delta x$ alltid gir et bedre svar. Det stemmer ikke på en datamaskin.

- Et stort $\Delta x$ gir **diskretiseringsfeil** fordi sekanten er en dårlig tilnærming til tangenten.
- Et svært lite $\Delta x$ kan gi **avrundingsfeil** fordi to nesten like flyttall trekkes fra hverandre.

Vi kan undersøke dette direkte.


In [ ]:
dx_values = np.logspace(-16, -1, 80)
exact = np.cos(1.0)

errors_forward = []
errors_central = []

for dx in dx_values:
    errors_forward.append(abs(forward_difference(np.sin, 1.0, dx) - exact))
    errors_central.append(abs(central_difference(np.sin, 1.0, dx) - exact))

plt.loglog(dx_values, errors_forward, label="Forward")
plt.loglog(dx_values, errors_central, label="Central")
plt.xlabel("dx")
plt.ylabel("Absolute error")
plt.legend()
plt.show()


## Numerisk derivasjon av eksperimentelle data

Det er her numerisk derivasjon virkelig blir nyttig. For målte data har vi ingen analytisk funksjon å derivere. Vi har bare målepunkter.

### Eksempel: ekvivalenspunkt fra en titrerkurve

Anta at vi har målt pH ved ulike tilsatte volum NaOH:


In [ ]:
volume = np.array([20.0, 22.0, 23.0, 24.0, 24.5, 25.0, 25.5, 26.0, 27.0, 28.0, 30.0])
pH = np.array([2.90, 3.20, 3.45, 3.85, 4.35, 7.00, 9.65, 10.15, 10.55, 10.80, 11.10])

plt.plot(volume, pH, "o-")
plt.xlabel("Added NaOH (mL)")
plt.ylabel("pH")
plt.show()


Vi kan beregne stigningen mellom hvert par av nabopunkter:

$$\frac{\Delta\mathrm{pH}}{\Delta V}
=\frac{\mathrm{pH}_{i+1}-\mathrm{pH}_i}{V_{i+1}-V_i}.$$

Denne stigningen hører **ikke** til bare det første eller det andre målepunktet. Den beskriver intervallet mellom dem. Derfor plasserer vi den deriverte i midtpunktet

$$V_{\mathrm{midt}}=\frac{V_i+V_{i+1}}{2}.$$

Dette er spesielt viktig når vi skal anslå ekvivalensvolumet. Merk at midtpunktet er summen av de to volumene delt på to, ikke $(V_{i+1}-V_i)/2$.


In [ ]:
dpH_dV = np.diff(pH) / np.diff(volume)
volume_mid = (volume[:-1] + volume[1:]) / 2

equivalence_index = np.argmax(dpH_dV)
equivalence_volume = volume_mid[equivalence_index]

plt.plot(volume_mid, dpH_dV, "o-")
plt.axvline(equivalence_volume, linestyle="--")
plt.xlabel("Added NaOH (mL)")
plt.ylabel("ΔpH / ΔV")
plt.show()

print(f"Estimated equivalence volume: {equivalence_volume:.2f} mL")


Med denne **førstedifferansen** får vi altså ett derivertestimat per intervall, og x-verdien er midtpunktet i intervallet.

### `np.gradient`

Når vi først har forstått differansene, kan NumPy gjøre arbeidet for oss:

`np.gradient(y, x)` beregner den deriverte på de **opprinnelige x-punktene**. I indre punkter brukes i praksis informasjon på begge sider (en sentral tilnærming), mens endepunktene må behandles ensidig.


In [ ]:
gradient = np.gradient(pH, volume)

eq_index_gradient = np.argmax(gradient)
eq_volume_gradient = volume[eq_index_gradient]

plt.plot(volume, gradient, "o-")
plt.axvline(eq_volume_gradient, linestyle="--")
plt.xlabel("Added NaOH (mL)")
plt.ylabel("dpH / dV")
plt.show()

print(f"Estimate with np.gradient: {eq_volume_gradient:.2f} mL")


De to metodene svarer ikke helt på samme måte fordi derivertestimatene er plassert forskjellig. Ved grove måleintervaller bør ekvivalensvolumet derfor ikke rapporteres med større presisjon enn dataene tillater.

```{admonition} Måleoppløsning
:class: important
Hvis ekvivalenspunktet er viktig, bør målingene ligge tettere rundt det bratte området. Numerisk behandling kan ikke skape informasjon som ikke finnes i måledataene.
```

### Derivasjon forsterker støy

En differanse bygger på forskjellen mellom nabomålinger. Små tilfeldige variasjoner kan derfor bli mye tydeligere etter derivasjon. Dette er en viktig grunn til å være forsiktig med numeriske deriverte av eksperimentelle data.

## Eksempel: reaksjonsfart fra konsentrasjonsdata


In [ ]:
time = np.array([0, 20, 40, 60, 80, 100], dtype=float)
concentration = np.array([1.00, 0.82, 0.68, 0.55, 0.45, 0.37])

rate_A = -np.gradient(concentration, time)

plt.plot(time, rate_A, "o-")
plt.xlabel("Time (s)")
plt.ylabel("Rate of disappearance of A (mol L$^{-1}$ s$^{-1}$)")
plt.show()


Her bruker vi minusfortegn fordi konsentrasjonen av reaktanten synker, mens reaksjonsfarten vanligvis oppgis som en positiv størrelse.

Du kan utforske titrerdataene direkte i Basthon:

<iframe src="../../basthon/?from=examples/numerical_titration_derivative.py" width="100%" height="650" frameborder="0" title="Basthon: numerisk derivasjon av titrerdata" loading="lazy" allowfullscreen></iframe>

## Oppgaver

```{admonition} Oppgave 1 – kjent derivert
:class: tip
Beregn $f'(1)$ numerisk for $f(x)=2x+2$, $f(x)=x^2$ og $f(x)=\sin x$. Sammenlikn med analytiske svar.
```

```{admonition} Oppgave 2 – tre differansemetoder
:class: tip
Implementer framover-, bakover- og sentraldifferansen selv. Undersøk feilen for $f(x)=e^x$ ved $x=1$ for flere verdier av $\Delta x$.
```

```{admonition} Oppgave 3 – optimal steglengde
:class: tip
Lag et log-log-plott av absolutt feil som funksjon av $\Delta x$. Forklar hvorfor feilen øker både for store og svært små steglengder.
```

```{admonition} Oppgave 4 – titrering
:class: tip
Bruk et titrerdatasett og beregn $\Delta\mathrm{pH}/\Delta V$. Plott den deriverte mot midtpunktene $(V_i+V_{i+1})/2$ og anslå ekvivalenspunktet.
```

```{admonition} Oppgave 5 – `np.gradient`
:class: tip
Gjør samme analyse med `np.gradient`. Sammenlikn plasseringen av den største deriverte med resultatet fra førsteforskjellene. Forklar hvorfor resultatene kan avvike litt.
```

```{admonition} Oppgave 6 – reaksjonsfart
:class: tip
Du har målt $[A]$ ved tidene `[0, 10, 20, 30, 40, 50]` s og fått `[0.800, 0.691, 0.598, 0.516, 0.447, 0.386]` M. Beregn reaksjonsfarten numerisk. Hvordan endrer farten seg med tiden?
```

```{admonition} Oppgave 7 – støy
:class: tip
Legg små tilfeldige avvik til et glatt konsentrasjonsdatasett og deriver både de opprinnelige og de støyende dataene. Hva skjer? Hvorfor bør numerisk derivasjon av eksperimentelle data tolkes med forsiktighet?
```
